# Acoustic Diagnosis & Design Challenge
### Building Acoustics — Engineering Physics

**One-week case study | Python / Google Colab**

Tujuan notebook: **model -> simulate -> diagnose -> compare -> decide**.

> Data pada notebook ini adalah data didaktik untuk latihan, bukan hasil pengukuran produk nyata.


## 1. Kasus dan target
Dua ruang kelas identik berukuran **8 × 6 × 3 m** dipisahkan partisi 12 m². Partisi terdiri dari dinding utama 10 m² dengan `R = 45 dB` dan pintu 2 m² dengan `R = 20 dB` pada 500 Hz.

Target proyek:
- `Req >= 40 dB`
- `T60 < 0.8 s`
- keputusan harus mempertimbangkan **weak element, flanking, sensitivity, dan cost index**.


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

C = 343.0

# Geometry
L, W, H = 8.0, 6.0, 3.0
V = L*W*H
S = 2*(L*W + L*H + W*H)

# Surface absorption at 500 Hz
surface_areas = np.array([48.0, 48.0, 84.0])  # floor, ceiling, walls
alpha_baseline = np.array([0.10, 0.10, 0.08])

# Partition
partition_areas = np.array([10.0, 2.0])        # wall, door
partition_R = np.array([45.0, 20.0])

TARGET_REQ = 40.0
TARGET_T60 = 0.8

print(f"V = {V:.1f} m^3, S = {S:.1f} m^2")


## 2. Fungsi model fisika


In [ ]:
def equivalent_absorption_area(areas, alphas):
    areas = np.asarray(areas, dtype=float)
    alphas = np.asarray(alphas, dtype=float)
    return np.sum(areas * alphas)

def rt60_sabine(V, A, c=C):
    return 24*np.log(10)*V/(c*A)

def rt60_eyring(V, S, alpha_bar, c=C):
    return (24*np.log(10)/c) * V/(-S*np.log(1-alpha_bar))

def composite_R(areas, R_db):
    areas = np.asarray(areas, dtype=float)
    R_db = np.asarray(R_db, dtype=float)
    tau_eq = np.sum(areas * 10**(-R_db/10)) / np.sum(areas)
    return -10*np.log10(tau_eq)

def combine_parallel_paths(*R_paths_db):
    tau = np.sum([10**(-R/10) for R in R_paths_db])
    return -10*np.log10(tau)


## 3. Baseline: reproduksi benchmark


In [ ]:
A0 = equivalent_absorption_area(surface_areas, alpha_baseline)
alpha_bar0 = A0/S
T0 = rt60_sabine(V, A0)
TE0 = rt60_eyring(V, S, alpha_bar0)
Req0 = composite_R(partition_areas, partition_R)

baseline = pd.Series({
    "V (m3)": V,
    "S (m2)": S,
    "A (m2)": A0,
    "alpha_bar": alpha_bar0,
    "T60 Sabine (s)": T0,
    "T60 Eyring (s)": TE0,
    "Req (dB)": Req0,
})
baseline


### Checkpoint reasoning
1. Target mana yang gagal?
2. Apakah meningkatkan `R` dinding utama saja pasti memperbaiki sistem secara signifikan?
3. Sebelum menjalankan sel berikutnya, tulis prediksi Anda.


## 4. Alternatif desain A–D


In [ ]:
def evaluate_case(wall_R=45, door_R=20, ceiling_alpha=0.10, floor_alpha=0.10):
    alphas = np.array([floor_alpha, ceiling_alpha, 0.08])
    A = equivalent_absorption_area(surface_areas, alphas)
    T = rt60_sabine(V, A)
    Req = composite_R(partition_areas, [wall_R, door_R])
    return A, T, Req

cases = {
    "Baseline": dict(wall_R=45, door_R=20, ceiling_alpha=0.10, floor_alpha=0.10, cost=0),
    "A: door 35 dB": dict(wall_R=45, door_R=35, ceiling_alpha=0.10, floor_alpha=0.10, cost=3),
    "B: wall 55 dB": dict(wall_R=55, door_R=20, ceiling_alpha=0.10, floor_alpha=0.10, cost=8),
    "C: A+B": dict(wall_R=55, door_R=35, ceiling_alpha=0.10, floor_alpha=0.10, cost=11),
    "D: ceiling alpha=0.60": dict(wall_R=45, door_R=20, ceiling_alpha=0.60, floor_alpha=0.10, cost=4),
}

rows=[]
for name, p in cases.items():
    A,T,Req=evaluate_case(p["wall_R"],p["door_R"],p["ceiling_alpha"],p["floor_alpha"])
    rows.append([name,A,T,Req,p["cost"],T<TARGET_T60,Req>=TARGET_REQ])

results=pd.DataFrame(rows, columns=["Case","A_m2","T60_s","Req_dB","CostIndex","Pass_T60","Pass_Req"])
results


In [ ]:
fig, ax = plt.subplots(figsize=(9,4.5))
ax.scatter(results["Req_dB"], results["T60_s"], s=80)
for _, row in results.iterrows():
    ax.annotate(row["Case"], (row["Req_dB"], row["T60_s"]), xytext=(5,5), textcoords="offset points")
ax.axvline(TARGET_REQ, linestyle="--", label="Target Req")
ax.axhline(TARGET_T60, linestyle="--", label="Target T60")
ax.set_xlabel("Composite reduction index, Req (dB)")
ax.set_ylabel("RT60 (s)")
ax.set_title("Trade-off map: insulation vs reverberation")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()


### Interpretasi wajib
Jelaskan mengapa alternatif B (`wall 45 -> 55 dB`) hampir tidak mengubah `Req`, sedangkan upgrade pintu memberikan perubahan besar. Gunakan **transmission coefficient**, bukan argumen verbal saja.


## 5. Sensitivity sweep — rating pintu


In [ ]:
door_sweep = np.arange(20, 46, 1)
Req_door = [composite_R(partition_areas, [45, Rdoor]) for Rdoor in door_sweep]

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(door_sweep, Req_door, marker="o", markersize=3)
ax.axhline(TARGET_REQ, linestyle="--", label="Target 40 dB")
ax.set_xlabel("Door reduction index R_door (dB)")
ax.set_ylabel("Composite Req (dB)")
ax.set_title("Sensitivity of composite insulation to door rating")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()


**Pertanyaan:** Pada kisaran mana peningkatan rating pintu mulai memberikan *diminishing return* karena elemen lain mulai dominan?


## 6. Sensitivity sweep — absorpsi plafon


In [ ]:
alpha_ceiling_sweep = np.linspace(0.10, 0.90, 81)
T_sweep=[]
for ac in alpha_ceiling_sweep:
    A = equivalent_absorption_area(surface_areas, [0.10, ac, 0.08])
    T_sweep.append(rt60_sabine(V, A))

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(alpha_ceiling_sweep, T_sweep)
ax.axhline(TARGET_T60, linestyle="--", label="Target T60 = 0.8 s")
ax.set_xlabel("Ceiling absorption coefficient")
ax.set_ylabel("T60 Sabine (s)")
ax.set_title("RT60 sensitivity to ceiling absorption")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()


## 7. Flanking: solusi yang tampak lulus bisa menjadi gagal


In [ ]:
Req_A = composite_R(partition_areas, [45,35])
flank_sweep = np.arange(40, 61, 1)
Rfield = np.array([combine_parallel_paths(Req_A, Rf) for Rf in flank_sweep])

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(flank_sweep, Rfield, marker="o", markersize=3)
ax.axhline(TARGET_REQ, linestyle="--", label="Target 40 dB")
ax.set_xlabel("Equivalent flanking path R_flank (dB)")
ax.set_ylabel("Combined field reduction index (dB)")
ax.set_title("Sensitivity to a parallel flanking path")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

pd.DataFrame({"R_flank_dB": flank_sweep, "R_combined_dB": Rfield}).head()


### Pertanyaan HOTS
1. Pada nilai `R_flank` berapa alternatif A kehilangan target 40 dB?
2. Mengapa indeks dB tidak boleh dijumlah atau dirata-ratakan secara langsung?
3. Apakah margin desain 1 dB cukup? Pertahankan jawaban dengan evidence dari sweep.


## 8. Multi-objective decision


In [ ]:
# Example scoring framework. Students may propose a better one, but must justify it.
def score(row):
    penalty = 0.0
    if row.Req_dB < TARGET_REQ:
        penalty += 10*(TARGET_REQ-row.Req_dB)
    if row.T60_s > TARGET_T60:
        penalty += 20*(row.T60_s-TARGET_T60)
    return row.CostIndex + penalty

results["IllustrativeScore"] = results.apply(score, axis=1)
results.sort_values("IllustrativeScore")


> **Jangan menjadikan skor di atas sebagai jawaban otomatis.** Bobot adalah asumsi desain. Jelaskan bagaimana rekomendasi berubah jika prioritas biaya, margin, atau flanking berubah.


## 9. Tantangan akhir mahasiswa


### Engineering decision
Pilih **maksimum dua intervensi**. Rekomendasi Anda harus menjawab:

1. mekanisme fisik dominan;
2. bukti numerik dari baseline dan sweep;
3. target yang dipenuhi dan margin;
4. risiko akibat flanking;
5. cost index;
6. satu kondisi yang dapat **membalik rekomendasi** Anda.

### Wajib diserahkan
- notebook yang dapat dijalankan ulang;
- minimal 3 grafik;
- tabel alternatif;
- engineering memo maksimal 2 halaman;
- bagian **Model Limitations**.


## 10. Model limitations — isi oleh mahasiswa


Tuliskan minimal empat keterbatasan. Pertimbangkan antara lain:
- satu pita 500 Hz;
- asumsi medan difus untuk Sabine/Eyring;
- data `R` dan `alpha` ideal/didaktik;
- flanking yang direpresentasikan sebagai satu jalur ekuivalen;
- tidak ada ketidakpastian pemasangan atau pengukuran;
- belum ada data sambungan dan spektrum lengkap.
